# Notebook 03: Model Training & Hyperparameter Tuning
## Project: Online Shopping Purchase Intention Prediction
**Primary Model**: Support Vector Machine (SVM)
**Comparison Model**: Decision Tree Classifier

### Methodology:
1. **Stratified 80/20 Train-Test Split**: The 20% test partition remains untouched throughout model tuning to ensure zero data leakage.
2. **Pipelines & ColumnTransformer**:
   - `StandardScaler` for numerical predictors (critical for SVM margin optimization).
   - `OneHotEncoder(handle_unknown='ignore')` for nominal features (`Month`, `VisitorType`).
   - Binary encoding for `Weekend`.
3. **Hyperparameter Tuning**: 5-Fold Stratified Cross-Validation maximizing **F1-Score**.


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

# 1. Load Data
df = pd.read_csv('../backend/data/online_shoppers_intention.csv')
X = df.drop(columns=['Revenue'])
X['Weekend'] = X['Weekend'].astype(int)
y = df['Revenue'].astype(int)

# 2. Stratified 80/20 Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
print(f"Training samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}")


### 3. SVM Pipeline & Hyperparameter Tuning
We optimize the regularization parameter $C$ and kernel coefficient $\gamma$ with an RBF kernel and `class_weight='balanced'`.


In [ ]:
num_cols = [
    'Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration',
    'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates',
    'PageValues', 'SpecialDay', 'OperatingSystems', 'Browser', 'Region', 'TrafficType'
]
cat_cols = ['Month', 'VisitorType']
bool_cols = ['Weekend']

preprocessor_svm = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
    ('bool', 'passthrough', bool_cols)
])

svm_pipeline = Pipeline([
    ('preprocessor', preprocessor_svm),
    ('classifier', SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=42))
])

svm_param_grid = {
    'classifier__C': [0.1, 1.0, 10.0, 100.0],
    'classifier__gamma': ['scale', 0.01, 0.001]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
svm_grid = GridSearchCV(svm_pipeline, svm_param_grid, cv=cv, scoring='f1', n_jobs=-1, verbose=1)
svm_grid.fit(X_train, y_train)

print(f"Best SVM Parameters: {svm_grid.best_params_}")
print(f"Best SVM 5-Fold CV F1: {svm_grid.best_score_:.4f}")


### 4. Decision Tree Pipeline & Hyperparameter Tuning
We optimize tree depth, leaf sample limits, and split criteria with `class_weight='balanced'`.


In [ ]:
preprocessor_dt = ColumnTransformer(transformers=[
    ('num', 'passthrough', num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
    ('bool', 'passthrough', bool_cols)
])

dt_pipeline = Pipeline([
    ('preprocessor', preprocessor_dt),
    ('classifier', DecisionTreeClassifier(class_weight='balanced', random_state=42))
])

dt_param_grid = {
    'classifier__max_depth': [3, 5, 7, 10, None],
    'classifier__min_samples_split': [2, 5, 10],
    'classifier__min_samples_leaf': [1, 2, 4],
    'classifier__criterion': ['gini', 'entropy']
}

dt_grid = GridSearchCV(dt_pipeline, dt_param_grid, cv=cv, scoring='f1', n_jobs=-1, verbose=1)
dt_grid.fit(X_train, y_train)

print(f"Best Decision Tree Parameters: {dt_grid.best_params_}")
print(f"Best Decision Tree 5-Fold CV F1: {dt_grid.best_score_:.4f}")
